In [7]:
pip install torch torchvision timm transformers scikit-learn matplotlib

Note: you may need to restart the kernel to use updated packages.


In [8]:
import sys
import torch

print("Python executable:", sys.executable)
print("PyTorch version:  ", torch.__version__)
print("CUDA available:   ", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "CUDA is not available. Please switch your Jupyter Kernel to 'breakhis_gpu' "
    "in the top right corner of the notebook."
)

device = torch.device("cuda")
print("Active GPU:       ", torch.cuda.get_device_name(0))

Python executable: C:\Users\User\miniconda3\envs\breakhis_gpu\python.exe
PyTorch version:   2.5.1+cu121
CUDA available:    True
Active GPU:        NVIDIA GeForce RTX 3070


In [9]:
import os
import zipfile
from pathlib import Path
from kaggle.api.kaggle_api_extended import KaggleApi

# Configuration Paths
BASE_DIR = Path(r"D:\vit_vlm\Breast_Cancer_Research")
DATA_DIR = BASE_DIR / "breakhis_dataset"
SAVE_DIR = BASE_DIR / "breakhis_savedmodels"

DATA_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle Token Setup
os.environ["KAGGLE_API_TOKEN"] = "KGAT_2a377ef3506fa3a701fe50a99049cf41"

api = KaggleApi()
api.authenticate()
print("Kaggle authenticated successfully.")

# Target dataset requested: waseemalastal/breakhis-breast-cancer-histopathological-dataset
dataset_slug = "waseemalastal/breakhis-breast-cancer-histopathological-dataset"

# Check if dataset already downloaded and extracted
existing_pngs = list(DATA_DIR.rglob("*.png"))
if len(existing_pngs) < 1000:
    print(f"Downloading {dataset_slug} to {DATA_DIR}...")
    api.dataset_download_files(dataset_slug, path=str(DATA_DIR), unzip=True)
    print("Download and extraction completed.")
else:
    print(f"BreaKHis dataset already detected locally ({len(existing_pngs)} images found).")

Kaggle authenticated successfully.
BreaKHis dataset already detected locally (15818 images found).


In [10]:
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def parse_breakhis_label(path: Path):
    name = path.name.lower()
    parts = [p.lower() for p in path.parts]
    joined = "/".join(parts)
    
    if any("malignant" in p for p in parts) or name.startswith("sob_m_") or "_m_" in name:
        return 1
    if any("benign" in p for p in parts) or name.startswith("sob_b_") or "_b_" in name:
        return 0
    return None

def extract_patient_id(path: Path):
    stem = path.stem
    # Standard BreaKHis naming convention: SOB_<M/B>_<Type>-<Year>-<SlideID>-<Mag>-<Seq>
    match = re.match(r"^(SOB_[A-Za-z]_[A-Za-z]+-\d+-[^-]+)-\d+-\\d+$", stem)
    if match:
        return match.group(1)
    parts = stem.split("-")
    if len(parts) >= 3 and parts[-1].isdigit() and parts[-2].isdigit():
        return "-".join(parts[:-2])
    return path.parent.name

rows = []
for p in DATA_DIR.rglob("*"):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
        label = parse_breakhis_label(p)
        if label is not None:
            rows.append({
                "path": str(p),
                "label": int(label),
                "patient_id": extract_patient_id(p),
                "filename": p.name
            })

df = pd.DataFrame(rows)
df = df.drop_duplicates(subset=["filename"]).reset_index(drop=True)
print(f"Total Images Found: {len(df)}")
print(f"Unique Patients:    {df['patient_id'].nunique()}")
print(f"Class Distribution: Benign(0)={sum(df['label']==0)} | Malignant(1)={sum(df['label']==1)}")

# Patient-Level Stratified Split (70% Train, 15% Val, 15% Test)
patient_df = df.drop_duplicates(subset=["patient_id"]).reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
splits = list(sgkf.split(patient_df, patient_df["label"], groups=patient_df["patient_id"]))

# Take 3 folds (~30%) for temp, remaining 7 (~70%) for train
temp_patients = set()
for fold in range(3):
    temp_patients.update(patient_df.iloc[splits[fold][1]]["patient_id"])

train_df = df[~df["patient_id"].isin(temp_patients)].reset_index(drop=True)
temp_df = df[df["patient_id"].isin(temp_patients)].reset_index(drop=True)

# Split temp 50/50 into Val and Test
temp_patient_df = temp_df.drop_duplicates(subset=["patient_id"]).reset_index(drop=True)
sgkf_sub = StratifiedGroupKFold(n_splits=2, shuffle=True, random_state=42)
v_idx, t_idx = next(sgkf_sub.split(temp_patient_df, temp_patient_df["label"], groups=temp_patient_df["patient_id"]))

val_patients = set(temp_patient_df.iloc[v_idx]["patient_id"])
test_patients = set(temp_patient_df.iloc[t_idx]["patient_id"])

val_df = temp_df[temp_df["patient_id"].isin(val_patients)].reset_index(drop=True)
test_df = temp_df[temp_df["patient_id"].isin(test_patients)].reset_index(drop=True)

print(f"\nTrain Set: {len(train_df)} images ({train_df['patient_id'].nunique()} patients)")
print(f"Val Set:   {len(val_df)} images ({val_df['patient_id'].nunique()} patients)")
print(f"Test Set:  {len(test_df)} images ({test_df['patient_id'].nunique()} patients)")

Total Images Found: 7909
Unique Patients:    82
Class Distribution: Benign(0)=2480 | Malignant(1)=5429

Train Set: 5323 images (57 patients)
Val Set:   1141 images (13 patients)
Test Set:  1445 images (12 patients)


In [11]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IMG_SIZE = 224
BATCH_SIZE = 16

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class BreaKHisDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["path"]).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(row["label"], dtype=torch.long)

train_loader = DataLoader(BreaKHisDataset(train_df, train_transform), batch_size=BATCH_SIZE, shuffle=True, pin_memory=False, num_workers=0)
val_loader   = DataLoader(BreaKHisDataset(val_df, eval_transform),   batch_size=BATCH_SIZE, shuffle=False, pin_memory=False, num_workers=0)
test_loader  = DataLoader(BreaKHisDataset(test_df, eval_transform),  batch_size=BATCH_SIZE, shuffle=False, pin_memory=False, num_workers=0)

# Compute inverse class frequencies to counteract class imbalance
class_counts = train_df["label"].value_counts().sort_index().values
weights = torch.tensor([sum(class_counts) / (len(class_counts) * c) for c in class_counts], dtype=torch.float).to(device)
criterion = torch.nn.CrossEntropyLoss(weight=weights, label_smoothing=0.05)

In [12]:
import timm
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

MODEL_CONFIG = {
    "vit": "vit_base_patch16_224",
    "swin": "swin_tiny_patch4_window7_224",
    "deit": "deit_base_patch16_224",
    "beit": "beit_base_patch16_224"
}

EPOCHS = 30
LR = 2e-5

for model_name, model_arch in MODEL_CONFIG.items():
    print(f"\n{'='*70}\nTRAINING {model_name.upper()} ({model_arch})\n{'='*70}")
    
    model = timm.create_model(model_arch, pretrained=True, num_classes=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.05)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
    scaler = torch.amp.GradScaler("cuda")
    
    best_val_f1 = 0.0
    save_path = SAVE_DIR / f"{model_name}_best.pth"
    
    for epoch in range(1, EPOCHS + 1):
        model.train()
        running_loss, total_train = 0.0, 0
        
       
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} [Train]", leave=False, dynamic_ncols=True):
            images = images.to(device)
            labels = labels.to(device)
            optimizer.zero_grad(set_to_none=True)
    
            with torch.amp.autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)
        
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
    
            running_loss += loss.item() * images.size(0)
            total_train += images.size(0)
            
        scheduler.step()
        train_loss = running_loss / total_train
        
        # Validation
        model.eval()
        v_preds, v_targets = [], []
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device, non_blocking=True)
                with torch.amp.autocast("cuda"):
                    outputs = model(images)
                preds = torch.argmax(outputs, dim=1).cpu().tolist()
                v_preds.extend(preds)
                v_targets.extend(labels.tolist())
                
        val_acc = accuracy_score(v_targets, v_preds)
        val_f1 = f1_score(v_targets, v_preds, zero_division=0)
        
        print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")
        
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), save_path)
            print(f"  --> Saved new best checkpoint: {save_path.name} (F1: {best_val_f1:.4f})")
            
    # Final Evaluation on Held-Out Test Set
    print(f"\n--- Testing Best Checkpoint for {model_name.upper()} ---")
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    
    t_preds, t_probs, t_targets = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast("cuda"):
                outputs = model(images)
            probs = torch.softmax(outputs, dim=1)[:, 1].cpu().tolist()
            preds = torch.argmax(outputs, dim=1).cpu().tolist()
            t_probs.extend(probs)
            t_preds.extend(preds)
            t_targets.extend(labels.tolist())
            
    print(f"Test Accuracy: {accuracy_score(t_targets, t_preds):.4f}")
    print(f"Test F1 Score: {f1_score(t_targets, t_preds, zero_division=0):.4f}")
    print(f"Test ROC-AUC:  {roc_auc_score(t_targets, t_probs):.4f}")
    print("Confusion Matrix:\n", confusion_matrix(t_targets, t_preds))
    
    del model, optimizer, scheduler, scaler
    torch.cuda.empty_cache()


TRAINING VIT (vit_base_patch16_224)


Epoch 01 | Train Loss: 0.3728 | Val Acc: 0.8344 | Val F1: 0.8737
  --> Saved new best checkpoint: vit_best.pth (F1: 0.8737)


Epoch 02 | Train Loss: 0.2356 | Val Acc: 0.8466 | Val F1: 0.8930
  --> Saved new best checkpoint: vit_best.pth (F1: 0.8930)


Epoch 03 | Train Loss: 0.1902 | Val Acc: 0.8054 | Val F1: 0.8671


Epoch 04 | Train Loss: 0.1696 | Val Acc: 0.8457 | Val F1: 0.8911


Epoch 05 | Train Loss: 0.1625 | Val Acc: 0.8431 | Val F1: 0.8922


... (119 outputs left out)

Epoch 29 | Train Loss: 0.1358 | Val Acc: 0.8133 | Val F1: 0.8711


Epoch 30 | Train Loss: 0.1358 | Val Acc: 0.8142 | Val F1: 0.8717

--- Testing Best Checkpoint for BEIT ---


C:\Users\User\AppData\Local\Temp\ipykernel_17316\2320697626.py:76: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(save_path, map_location=dev

Test Accuracy: 0.9474
Test F1 Score: 0.9633
Test ROC-AUC:  0.9759
Confusion Matrix:
 [[372  34]
 [ 42 997]]


In [14]:
from pathlib import Path

SAVE_DIR = Path(r"D:\vit_vlm\Breast_Cancer_Research\breakhis_savedmodels")
MODELS = ["vit", "swin", "deit", "beit"]

print(f"{'='*60}\nCHECKPOINT INTEGRITY & EXISTENCE AUDIT\n{'='*60}")
missing = []

for m in MODELS:
    ckpt = SAVE_DIR / f"{m}_best.pth"
    if ckpt.exists() and ckpt.stat().st_size > 0:
        mb = ckpt.stat().st_size / (1024 * 1024)
        print(f"[FOUND]     {m.upper():<6} -> {ckpt.name:<18} ({mb:.2f} MB)")
    else:
        print(f"[NOT FOUND] {m.upper():<6} -> Expected: {ckpt.name}")
        missing.append(m)

print(f"{'-'*60}")
assert not missing, f"Missing checkpoints for: {missing}. Resolve before evaluation."
print(f"All {len(MODELS)} model checkpoints are verified and ready for evaluation.")

CHECKPOINT INTEGRITY & EXISTENCE AUDIT
[FOUND]     VIT    -> vit_best.pth       (327.36 MB)
[FOUND]     SWIN   -> swin_best.pth      (105.05 MB)
[FOUND]     DEIT   -> deit_best.pth      (327.36 MB)
[FOUND]     BEIT   -> beit_best.pth      (327.23 MB)
------------------------------------------------------------
All 4 model checkpoints are verified and ready for evaluation.


In [15]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import timm
import torch
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)

# 1. Paths and device configuration
SAVE_DIR = Path(r"D:\vit_vlm\Breast_Cancer_Research\breakhis_savedmodels")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 2. Model dictionary mapping keys to timm architecture names
MODEL_CONFIG = {
    "vit": "vit_base_patch16_224",
    "swin": "swin_tiny_patch4_window7_224",
    "deit": "deit_base_patch16_224",
    "beit": "beit_base_patch16_224",
}

BINARY_CLASSES = ["Benign", "Malignant"]

# 3. Iterate through each model checkpoint
for model_name, model_arch in MODEL_CONFIG.items():
    ckpt_path = SAVE_DIR / f"{model_name}_best.pth"

    if not ckpt_path.exists():
        print(f"\nSkipping {model_name.upper()}: Checkpoint not found at {ckpt_path}")
        continue

    print(f"\n{'='*60}")
    print(f"Generating Evaluation Plots for: {model_name.upper()}")
    print(f"{'='*60}")

    # Instantiate model architecture with binary head
    model = timm.create_model(model_arch, pretrained=False, num_classes=2)

    # Handle both direct state_dict and wrapped checkpoint dictionaries
    checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        model.load_state_dict(checkpoint["state_dict"])
    else:
        model.load_state_dict(checkpoint)

    model = model.to(device)
    model.eval()

    all_targets = []
    all_preds = []
    all_probs = []

    # Inference loop
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(images)

            # Extract malignant probability (index 1) and predicted class
            probs = torch.softmax(logits, dim=1)[:, 1]
            preds = torch.argmax(logits, dim=1)

            all_targets.extend(labels.cpu().numpy().tolist())
            all_preds.extend(preds.cpu().numpy().tolist())
            all_probs.extend(probs.cpu().numpy().tolist())

    y_true = np.array(all_targets)
    y_pred = np.array(all_preds)
    y_prob = np.array(all_probs)

    # -------------------------------------------------------------
    # A. Confusion Matrix (Purple Palette)
    # -------------------------------------------------------------
    cm = confusion_matrix(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(6, 5))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm, display_labels=BINARY_CLASSES
    )
    # "Purples" provides a purple sequential gradient
    disp.plot(cmap="Purples", ax=ax, values_format="d", colorbar=True)

    ax.set_title(
        f"{model_name.upper()} - Binary Confusion Matrix",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )
    plt.tight_layout()

    cm_save_path = SAVE_DIR / f"{model_name}_confusion_matrix.png"
    plt.savefig(cm_save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved Confusion Matrix: {cm_save_path.name}")

    # -------------------------------------------------------------
    # B. ROC-AUC Curve
    # -------------------------------------------------------------
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_val = roc_auc_score(y_true, y_prob)

    fig, ax = plt.subplots(figsize=(6, 5))
    # Line plotted in deep purple (#6a0dad)
    ax.plot(
        fpr,
        tpr,
        color="#6a0dad",
        lw=2.5,
        label=f"{model_name.upper()} (AUC = {auc_val:.4f})",
    )
    ax.plot(
        [0, 1],
        [0, 1],
        color="grey",
        lw=1.5,
        linestyle="--",
        label="Chance (AUC = 0.5000)",
    )

    ax.set_xlim([0.0, 1.0])
    ax.set_ylim([0.0, 1.05])
    ax.set_xlabel("False Positive Rate (1 - Specificity)", fontsize=11)
    ax.set_ylabel("True Positive Rate (Sensitivity)", fontsize=11)
    ax.set_title(
        f"{model_name.upper()} - ROC Curve",
        fontsize=13,
        fontweight="bold",
        pad=12,
    )
    ax.legend(loc="lower right", fontsize=10)
    ax.grid(True, linestyle=":", alpha=0.6)
    plt.tight_layout()

    roc_save_path = SAVE_DIR / f"{model_name}_roc_auc.png"
    plt.savefig(roc_save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved ROC Curve:        {roc_save_path.name}")

    # Free memory between models
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll model plots have been generated and saved.")

Using device: cuda

Generating Evaluation Plots for: VIT
Saved Confusion Matrix: vit_confusion_matrix.png
Saved ROC Curve:        vit_roc_auc.png

Generating Evaluation Plots for: SWIN
Saved Confusion Matrix: swin_confusion_matrix.png
Saved ROC Curve:        swin_roc_auc.png

Generating Evaluation Plots for: DEIT
Saved Confusion Matrix: deit_confusion_matrix.png
Saved ROC Curve:        deit_roc_auc.png

Generating Evaluation Plots for: BEIT
Saved Confusion Matrix: beit_confusion_matrix.png
Saved ROC Curve:        beit_roc_auc.png

All model plots have been generated and saved.
